# The `HODDIES` backend

[`HODDIES`](https://github.com/antoine-rocher/HODDIES) is a HOD implementation
that populates a halo catalog with central and satellite galaxies. The
`acm.catalogs.backends.hoddies.HODDIESBackend` wraps it and accepts halo
catalogs and particle data **in memory**, so it can be used with arbitrary
simulation outputs.

Requirements:

```bash
pip install -e ".[hoddies]"
```

> **Note**: the `hoddies` extra is temporarily pinned to a fork commit that
> fixes HODDIES issue #3 (custom halo catalogs). See `pyproject.toml`.


In [ ]:
import numpy as np
import pandas as pd

from acm.catalogs.backends.hoddies import HODDIESBackend
from acm.catalogs.dataclasses import Tracer


## 1. Build a synthetic halo catalog

Only the columns used by HODDIES are needed:

| column | meaning | unit |
|---|---|---|
| `x, y, z` | halo centre | Mpc/h |
| `vx, vy, vz` | halo velocity | km/s |
| `Mh` | halo mass | Msun/h |
| `Rh` | halo radius | kpc/h |
| `Rs` | NFW scale radius | kpc/h |
| `c` | concentration | - |
| `Vrms` | particle velocity dispersion | km/s |
| `halo_id` | unique halo id | - |

`Rs` and `Vrms` are optional for NFW satellites; `log10_Mh` is derived by
HODDIES.


In [ ]:
rng = np.random.default_rng(42)
n_halo = 200
boxsize = 500.0  # Mpc/h

Rh = rng.uniform(300.0, 1500.0, n_halo)  # kpc/h
c = rng.uniform(3.0, 10.0, n_halo)
Rs = Rh / c  # kpc/h

halo_catalog = {
    "x": rng.uniform(0.0, boxsize, n_halo),
    "y": rng.uniform(0.0, boxsize, n_halo),
    "z": rng.uniform(0.0, boxsize, n_halo),
    "vx": rng.normal(0.0, 200.0, n_halo),
    "vy": rng.normal(0.0, 200.0, n_halo),
    "vz": rng.normal(0.0, 200.0, n_halo),
    "Mh": 10.0 ** rng.uniform(12.5, 15.0, n_halo),  # Msun/h
    "Rh": Rh,
    "Rs": Rs,
    "c": c,
    "Vrms": rng.uniform(100.0, 500.0, n_halo),  # km/s
    "halo_id": np.arange(n_halo),
}

pd.DataFrame(halo_catalog).head()


## 2. Populate the halos with galaxies

The backend uses the `LRG` tracer template from the HODDIES default
configuration. `Tracer.params` overrides individual HOD parameters;
`density=None` disables the target number-density rescaling of `Ac`.

With `use_particles=False`, satellites are drawn from an NFW profile.


In [ ]:
tracer = Tracer(
    name="LRG",
    params={
        "log_Mcent": 13.0,
        "sigma_M": 0.2,
        "M_0": 13.0,
        "M_1": 13.5,
        "alpha": 1.0,
        "f_sigv": 0.8,
        "density": None,
    },
)

backend = HODDIESBackend(
    hcat={"boxsize": boxsize},
    tracers=["LRG"],
    use_particles=False,
)
backend.load_dark_matter_catalog(redshift=0.5, hcat=halo_catalog)

galaxy_catalog = backend.make_galaxy_catalog(
    backend.get_dark_matter_catalog(redshift=0.5),
    [tracer],
    seed=42,
    verbose=False,
)[tracer]

print(galaxy_catalog.shape)
galaxy_catalog.head()


## 3. Optional: satellites from particle data

To position satellites with simulation particles, pass a particle catalog
containing `x, y, z, vx, vy, vz, halo_id` and construct the backend with
`use_particles=True`. Particles are sampled from the host halo when enough are
available; otherwise the remaining satellites fall back to NFW.


In [ ]:
n_particle_per_halo = 20
n_particle = n_halo * n_particle_per_halo

particles = {
    "x": rng.uniform(0.0, boxsize, n_particle),
    "y": rng.uniform(0.0, boxsize, n_particle),
    "z": rng.uniform(0.0, boxsize, n_particle),
    "vx": rng.normal(0.0, 200.0, n_particle),
    "vy": rng.normal(0.0, 200.0, n_particle),
    "vz": rng.normal(0.0, 200.0, n_particle),
    "halo_id": np.repeat(np.arange(n_halo), n_particle_per_halo),
}

backend_particles = HODDIESBackend(
    hcat={"boxsize": boxsize},
    tracers=["LRG"],
    use_particles=True,
)
backend_particles.load_dark_matter_catalog(
    redshift=0.5,
    hcat=halo_catalog,
    particles=particles,
)

galaxy_catalog_particles = backend_particles.make_galaxy_catalog(
    backend_particles.get_dark_matter_catalog(redshift=0.5),
    [tracer],
    seed=42,
    verbose=False,
)[tracer]

print(galaxy_catalog_particles.shape)
galaxy_catalog_particles.head()


## 4. Measure the galaxy power spectrum

The `acm` estimators accept plain Cartesian positions. HODDIES returns
positions in `[0, boxsize)`, so we wrap them and centre the box on zero before
passing them to the mesh.


In [ ]:
from acm.estimators.galaxy_clustering.spectrum import PowerSpectrumMultipoles

positions = galaxy_catalog[["x", "y", "z"]].to_numpy()
positions = (positions % boxsize) - boxsize / 2.0

estimator = PowerSpectrumMultipoles(
    backend="jaxpower",
    data_positions=positions,
    boxsize=boxsize,
    boxcenter=0.0,
    meshsize=128,
)
estimator.backend.set_density_contrast()
result = estimator.compute(edges={"step": 0.01}, ells=(0, 2), los="z")
estimator.plot(result)


## Notes

- HODDIES expects positions in Mpc/h, velocities in km/s, masses in Msun/h and
  `Rh`/`Rs` in kpc/h.
- The output DataFrame always contains `x, y, z, vx, vy, vz, mass, id, is_cent`
  (`mass` is the host-halo mass and `id` the host-halo id).
- Particle `halo_id` values should be compact, non-negative integers.
- HODDIES adds a `log10_Mh` column to the halo catalog. Passing an
  `mpytools.Catalog` modifies that object in place.
- The power-spectrum estimator requires `jaxpower`
  (`pip install -e ".[estimators]"`).
- Particle satellites use HODDIES' native generic assignment path. The
  `hoddies` extra pins a fork commit with the halo-wise assignment fix until
  it is available upstream.
